In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q14/q14_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Define promotion type filter
p_type_like = "PROMO"

# Filter lineitem by ship date and select needed columns
flineitem = lineitem.loc[
    lineitem.L_SHIPDATE.between("1994-03-01", "1994-04-01", inclusive="left"),
    ["L_EXTENDEDPRICE", "L_DISCOUNT", "L_PARTKEY"]
]

# Compute revenue per line
flineitem["TMP"] = flineitem.L_EXTENDEDPRICE * (1.0 - flineitem.L_DISCOUNT)

# Total revenue
total_rev = flineitem.TMP.sum()

# Promo part keys
promo_keys = part.P_PARTKEY[part.P_TYPE.str.startswith(p_type_like)]

# Promo revenue
promo_rev = flineitem.TMP[flineitem.L_PARTKEY.isin(promo_keys)].sum()

# Final metric
total = promo_rev * 100 / total_rev